## tl;dr

This notebook performs a **read-only** audit of the existing U.S. macro database and the frozen main-page representation map. It verifies two decision-critical findings used in the report:

1. the catalog contains 45 raw series, of which 36 are represented on the current main page and 9 are not;
2. row-based 12-period lags misstate CPI and core CPI year-over-year changes after a missing calendar month.

No database rows, service files, or application outputs are modified.

## Context & Methods

Audit date: **2026-08-11**. The notebook locates the repository dynamically, opens `pboc_data.db` with SQLite URI `mode=ro`, and runs bounded aggregate queries. The representation set is a frozen code-review parameter for commit `7b7656f`: it includes series shown directly in charts, raw inputs used by derived main-page series, and the `FEDFUNDS` headline card.

The calendar-alignment check compares the current implementation's 12-row lag with an exact one-year date join. A nonzero difference is treated as a transformation error, not as a data revision or forecast difference.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'server.py').is_file() and (candidate / 'services').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate repository root from notebook working directory.')

repo_root = find_repo_root(Path.cwd().resolve())
database_path = repo_root / 'pboc_data.db'
connection = sqlite3.connect(f'{database_path.as_uri()}?mode=ro', uri=True)
connection.execute('PRAGMA query_only = ON')

pd.DataFrame([{
    'database': database_path.name,
    'open_mode': 'read-only',
    'audit_date': '2026-08-11',
    'code_commit': '7b7656f',
}])

,database,open_mode,audit_date,code_commit
0,pboc_data.db,read-only,2026-08-11,7b7656f


## Data

The first query checks table scope, freshness, missing provenance fields, source-status labels, and duplicate indicator-period keys. The second query shows the current provenance classification distribution.

In [2]:
quality_sql = '''
SELECT
    COUNT(*) AS observation_rows,
    COUNT(DISTINCT indicator_code) AS distinct_series,
    MIN(period) AS first_period,
    MAX(period) AS last_period,
    MAX(updated_at) AS latest_database_update,
    SUM(CASE WHEN source_url IS NULL OR TRIM(source_url) = '' THEN 1 ELSE 0 END) AS missing_source_url_rows,
    SUM(CASE WHEN data_status <> 'official' OR data_status IS NULL THEN 1 ELSE 0 END) AS non_official_status_rows,
    (
        SELECT COUNT(*) FROM (
            SELECT indicator_code, period
            FROM us_macro_observations
            GROUP BY indicator_code, period
            HAVING COUNT(*) > 1
        )
    ) AS duplicate_indicator_period_keys
FROM us_macro_observations;
'''
quality_profile = pd.read_sql_query(quality_sql, connection)
display(quality_profile)

source_profile = pd.read_sql_query(
    '''
    SELECT source_name, source_type, data_status, COUNT(*) AS observation_rows,
           COUNT(DISTINCT indicator_code) AS distinct_series
    FROM us_macro_observations
    GROUP BY source_name, source_type, data_status
    ORDER BY observation_rows DESC
    ''',
    connection,
)
display(source_profile)

,observation_rows,distinct_series,first_period,last_period,latest_database_update,missing_source_url_rows,non_official_status_rows,duplicate_indicator_period_keys
0,183365,45,1854-12-01,2026-08-11,2026-08-11T17:45:28.257936,0,0,0


,source_name,source_type,data_status,observation_rows,distinct_series
0,FRED (Federal Reserve Bank of St. Louis),us_macro,official,183365,45


## Results

### Main-page representation

The following parameter is intentionally explicit and reviewable. It should be updated only when the UI/service representation changes, so future executions can reveal coverage drift instead of silently redefining the baseline.

In [3]:
represented_codes = {
    'A091RC1Q027SBEA', 'CES0500000003', 'CIVPART', 'CPIAUCSL', 'CPILFESL',
    'DFF', 'DGS10', 'DGS2', 'DGS30', 'DGS3MO', 'DGS5', 'DPRIME', 'EMRATIO',
    'FEDFUNDS', 'FGEXPND', 'FGRECPT', 'FYFSD', 'GDPC1', 'GFDEBTN',
    'GFDEGDQ188S', 'ICSA', 'INDPRO', 'JTSHIR', 'JTSJOL', 'JTSLDL', 'JTSQUR',
    'MORTGAGE30US', 'OPHNFB', 'PAYEMS', 'PCEPILFE', 'RSAFS', 'SOFR',
    'T10Y2Y', 'T10YIE', 'U6RATE', 'UNRATE',
}

catalog = pd.read_sql_query(
    'SELECT DISTINCT indicator_code, indicator_name FROM us_macro_observations ORDER BY indicator_code',
    connection,
)
catalog['status'] = catalog['indicator_code'].map(
    lambda code: '主页面已呈现' if code in represented_codes else '主页面未呈现'
)
coverage = (
    catalog.groupby('status', as_index=False)
    .agg(series_count=('indicator_code', 'nunique'))
    .assign(share_pct=lambda frame: (100 * frame['series_count'] / catalog['indicator_code'].nunique()).round(1))
    .sort_values('series_count', ascending=False)
)
display(coverage)

not_on_main_page = catalog.loc[
    catalog['status'].eq('主页面未呈现'),
    ['indicator_code', 'indicator_name'],
].reset_index(drop=True)
display(not_on_main_page)

assert catalog['indicator_code'].nunique() == 45
assert len(represented_codes) == 36
assert len(not_on_main_page) == 9

,status,series_count,share_pct
0,主页面已呈现,36,80.0
1,主页面未呈现,9,20.0


,indicator_code,indicator_name
0,DTWEXBGS,广义美元指数
1,JTSJOR,JOLTS 职位空缺率
2,M2SL,美国 M2 货币供应
3,QUSR628BIS,BIS 美国实际住宅价格指数（2010=100）
4,T10Y3M,10Y－3M 国债期限利差
5,UMCSENT,密歇根大学消费者信心指数
6,USREC,NBER 衰退期标记
7,VIXCLS,VIX 波动率指数
8,WALCL,美联储总资产


### CPI calendar alignment

The raw CPI and core CPI series have no observation for `2025-10-01`. Because a 12-row lag assumes a complete monthly grid, every later row remains misaligned until enough rows restore the original calendar offset. The query below calculates both methods for all eligible months and then isolates the latest reviewed comparison period.

In [4]:
missing_month_check = pd.read_sql_query(
    '''
    SELECT indicator_code,
           SUM(CASE WHEN period = '2025-10-01' THEN 1 ELSE 0 END) AS october_2025_rows
    FROM us_macro_observations
    WHERE indicator_code IN ('CPIAUCSL', 'CPILFESL')
    GROUP BY indicator_code
    ORDER BY indicator_code
    ''',
    connection,
)
display(missing_month_check)

alignment_sql = '''
WITH lagged AS (
    SELECT
        indicator_code,
        period,
        value,
        LAG(value, 12) OVER (PARTITION BY indicator_code ORDER BY period) AS row_lag_value
    FROM us_macro_observations
    WHERE indicator_code IN ('CPIAUCSL', 'CPILFESL')
), paired AS (
    SELECT
        current.indicator_code,
        current.period,
        current.value,
        current.row_lag_value,
        prior.value AS calendar_lag_value
    FROM lagged AS current
    LEFT JOIN us_macro_observations AS prior
      ON prior.indicator_code = current.indicator_code
     AND prior.period = DATE(current.period, '-1 year')
)
SELECT
    indicator_code,
    period,
    100.0 * (value / row_lag_value - 1.0) AS row_lag_yoy_pct,
    100.0 * (value / calendar_lag_value - 1.0) AS calendar_yoy_pct,
    100.0 * (value / row_lag_value - value / calendar_lag_value) AS difference_pp
FROM paired
WHERE row_lag_value IS NOT NULL AND calendar_lag_value IS NOT NULL
ORDER BY indicator_code, period;
'''
alignment = pd.read_sql_query(alignment_sql, connection)
alignment['abs_difference_pp'] = alignment['difference_pp'].abs()

mismatch_summary = (
    alignment.loc[alignment['abs_difference_pp'] > 1e-10]
    .groupby('indicator_code', as_index=False)
    .agg(
        mismatched_periods=('period', 'count'),
        first_mismatch=('period', 'min'),
        last_mismatch=('period', 'max'),
        max_abs_difference_pp=('abs_difference_pp', 'max'),
    )
)
display(mismatch_summary)

latest = (
    alignment.loc[alignment['period'].eq('2026-06-01'), [
        'indicator_code', 'period', 'row_lag_yoy_pct', 'calendar_yoy_pct', 'difference_pp'
    ]]
    .round(4)
    .reset_index(drop=True)
)
display(latest)

assert missing_month_check['october_2025_rows'].eq(0).all()
assert mismatch_summary.set_index('indicator_code')['mismatched_periods'].to_dict() == {
    'CPIAUCSL': 8,
    'CPILFESL': 8,
}
assert latest.shape[0] == 2

,indicator_code,october_2025_rows
0,CPIAUCSL,0
1,CPILFESL,0


,indicator_code,mismatched_periods,first_mismatch,last_mismatch,max_abs_difference_pp
0,CPIAUCSL,8,2025-11-01,2026-06-01,0.437478
1,CPILFESL,8,2025-11-01,2026-06-01,0.437309


,indicator_code,period,row_lag_yoy_pct,calendar_yoy_pct,difference_pp
0,CPIAUCSL,2026-06-01,3.7265,3.4635,0.263
1,CPILFESL,2026-06-01,2.8058,2.5658,0.240


## Takeaways

- The U.S. macro catalog is already substantial enough to justify product work before broad source expansion.
- Main-page coverage is incomplete but recoverable from existing data: 9 cataloged series are currently absent from that page.
- The CPI issue is reproducible and material: both reviewed series show eight mismatched periods after the missing October 2025 row.
- The safe implementation gate is calendar-keyed change calculation with explicit missing-period behavior, followed by regression tests and reconciliation between `/us-macro` and `/macro-analytics`.
- All current observations have `data_status='official'` under a single FRED source name, while `source_type='us_macro'` is only a domain label. Source authority and vintage metadata therefore need a more expressive schema before the module is used for cross-country research.

In [5]:
connection.close()
print('Read-only audit complete; SQLite connection closed.')

Read-only audit complete; SQLite connection closed.
